# Premiers pas avec un LLM mocké

**Objectif** : appeler un « LLM » sans clé d'API, sans réseau et sans coût.

**Pourquoi c'est utile en LLMOps ?**
Un vrai LLM est "lent", payant et non déterministe. Pour tester notre code, il faut un
« faux modèle » qui répond toujours la même chose, instantanément et gratuitement.

**Outil** : LiteLLM, une bibliothèque qui appelle +100 fournisseurs avec la même syntaxe.
Son paramètre `mock_response` renvoie une réponse sans jamais contacter le modèle.

In [ ]:
!pip install litellm langfuse dotenv

   ---------------------------------------- 0.0/637.5 kB ? eta -:--:--
   ---------------------------------------- 637.5/637.5 kB 4.6 MB/s  0:00:00

   ----------------------------------------  0/12 [wrapt]
   ----------------------------------------  0/12 [wrapt]
   ----------------------------------------  0/12 [wrapt]
   ----------------------------------------  0/12 [wrapt]
   ----------------------------------------  0/12 [wrapt]
   ----------------------------------------  0/12 [wrapt]
   ----------------------------------------  0/12 [wrapt]
   ----------------------------------------  0/12 [wrapt]
   --- ------------------------------------  1/12 [protobuf]
   --- ------------------------------------  1/12 [protobuf]
   --- ------------------------------------  1/12 [protobuf]
   --- ------------------------------------  1/12 [protobuf]
   --- ------------------------------------  1/12 [protobuf]
   --- ------------------------------------  1/12 [protobuf]
   ------ -----------

Import

In [ ]:
import litellm
from litellm import completion

litellm.suppress_debug_info = True  # masque les messages promotionnels de LiteLLM

In [6]:
reponse = completion(
    model="openai/gpt-4o-mini",  # ce modèle n'est JAMAIS appelé
    messages=[{"role": "user", "content": "Bonjour, qui es-tu ?"}],
    mock_response="Je suis l'assistant de <votre nom>.",
)

print(reponse.choices[0].message.content)

Je suis l'assistant de <votre nom>.


In [ ]:
# La réponse mockée a EXACTEMENT le même format qu'une vraie réponse OpenAI :
# le code de l'application ne voit pas la différence.
print("Type          :", type(reponse).__name__)
print("Modèle        :", reponse.model)
print("Contenu       :", reponse.choices[0].message.content)
print("finish_reason :", reponse.choices[0].finish_reason)  # "stop" = réponse terminée normalement
print("Tokens        :", reponse.usage)

Type          : ModelResponse
Modèle        : gpt-4o-mini
Contenu       : Je suis l'assistant de <votre nom>.
finish_reason : stop
Tokens        : Usage(completion_tokens=20, prompt_tokens=10, total_tokens=30, completion_tokens_details=None, prompt_tokens_details=None)


In [17]:
# Les chatbots affichent la réponse mot par mot : c'est le streaming.
# Le mock sait le simuler : il découpe la réponse en morceaux ("chunks").
flux = completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Explique le RAG en une phrase."}],
    mock_response="Le RAG consiste à retrouver des documents pertinents, puis à les donner "
                  "au LLM pour qu'il réponde à partir d'eux plutôt que de sa mémoire.",
    stream=True,
)

morceaux = []
for chunk in flux:
    texte = chunk.choices[0].delta.content or ""  # le dernier chunk est vide (None)
    morceaux.append(texte)
    print(texte, end="", flush=True)

print(f"\n\n→ {len(morceaux)} morceaux reçus")

Le RAG consiste à retrouver des documents pertinents, puis à les donner au LLM pour qu'il réponde à partir d'eux plutôt que de sa mémoire.

→ 47 morceaux reçus


In [ ]:
# 🧪 À vous : affichez chaque morceau entre crochets pour voir le découpage.

In [20]:
# Solution courte : on réutilise la liste `morceaux` remplie à la cellule 4.
# ⚠️ On ne peut PAS refaire une boucle sur `flux` : un flux se lit une seule fois (c'est un générateur).
print(" ".join(f"[{m}]" for m in morceaux))

# Solution détaillée : on relance un flux et on inspecte chaque chunk.
flux = completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Explique le RAG en une phrase."}],
    mock_response="Le RAG consiste à retrouver des documents pertinents, puis à les donner "
                  "au LLM pour qu'il réponde à partir d'eux plutôt que de sa mémoire.",
    stream=True,
)
for i, chunk in enumerate(flux):
    choix = chunk.choices[0]
    print(f"{i:>3} | [{choix.delta.content}]")

[Le ] [RAG] [ co] [nsi] [ste] [ à ] [ret] [rou] [ver] [ de] [s d] [ocu] [men] [ts ] [per] [tin] [ent] [s, ] [pui] [s à] [ le] [s d] [onn] [er ] [au ] [LLM] [ po] [ur ] [qu'] [il ] [rép] [ond] [e à] [ pa] [rti] [r d] ['eu] [x p] [lut] [ôt ] [que] [ de] [ sa] [ mé] [moi] [re.] []
  0 | [Le ]
  1 | [RAG]
  2 | [ co]
  3 | [nsi]
  4 | [ste]
  5 | [ à ]
  6 | [ret]
  7 | [rou]
  8 | [ver]
  9 | [ de]
 10 | [s d]
 11 | [ocu]
 12 | [men]
 13 | [ts ]
 14 | [per]
 15 | [tin]
 16 | [ent]
 17 | [s, ]
 18 | [pui]
 19 | [s à]
 20 | [ le]
 21 | [s d]
 22 | [onn]
 23 | [er ]
 24 | [au ]
 25 | [LLM]
 26 | [ po]
 27 | [ur ]
 28 | [qu']
 29 | [il ]
 30 | [rép]
 31 | [ond]
 32 | [e à]
 33 | [ pa]
 34 | [rti]
 35 | [r d]
 36 | ['eu]
 37 | [x p]
 38 | [lut]
 39 | [ôt ]
 40 | [que]
 41 | [ de]
 42 | [ sa]
 43 | [ mé]
 44 | [moi]
 45 | [re.]
 46 | [None]


In [ ]:
# En production, les fournisseurs renvoient des erreurs : 429 (trop de requêtes), 500, timeouts…
# On passe une EXCEPTION à mock_response pour tester notre gestion d'erreur sans attendre une vraie panne.
panne = litellm.RateLimitError(
    message="Quota dépassé (simulé)", llm_provider="openai", model="gpt-4o-mini"
)


def repondre(question: str, **kwargs) -> str:
    """Notre mini-application : elle doit rester polie même quand le LLM tombe."""
    try:
        r = completion(
            model="openai/gpt-4o-mini",
            messages=[{"role": "user", "content": question}],
            **kwargs,
        )
        return r.choices[0].message.content
    except litellm.RateLimitError:
        return "⏳ Le service est surchargé, réessayez dans quelques instants."


print("Cas normal :", repondre("Qu'est-ce qu'un token ?",
                               mock_response="Un token est un morceau de mot, l'unité que lit un LLM."))
print("Cas panne  :", repondre("Qu'est-ce qu'un token ?", mock_response=panne))

Cas normal : Un token est un morceau de mot, l'unité que lit un LLM.
Cas panne  : ⏳ Le service est surchargé, réessayez dans quelques instants.


In [24]:
# 🧪 À vous : simulez un litellm.Timeout. Est-il géré par repondre() ? Corrigez la fonction.

timeout = litellm.Timeout(message="Le modèle n'a pas répondu à temps (simulé)",
                          model="gpt-4o-mini", llm_provider="openai")
try:
    repondre("Qu'est-ce qu'un token ?", mock_response=timeout)
except litellm.Timeout as e:
    print("💥 Non géré : l'application plante avec", type(e).__name__)


# Nouvelle fonction

def repondre(question: str, **kwargs) -> str:
    """Notre mini-application : elle doit rester polie même quand le LLM tombe."""
    try:
        r = completion(
            model="openai/gpt-4o-mini",
            messages=[{"role": "user", "content": question}],
            **kwargs,
        )
        return r.choices[0].message.content
    except litellm.RateLimitError:          # 429 : temporaire, on peut réessayer
        return "⏳ Le service est surchargé, réessayez dans quelques instants."
    except litellm.Timeout:                 # trop lent : temporaire aussi
        return "⌛ Le modèle met trop de temps à répondre, réessayez."


# On teste les 4 branches en quelques millisecondes, sans aucune vraie panne
cas = {
    "normal ": "Un token est un morceau de mot.",
    "429    ": litellm.RateLimitError(message="quota", llm_provider="openai", model="gpt-4o-mini"),
    "timeout": timeout,
}
for nom, mock in cas.items():
    print(f"{nom} → {repondre('Qu est-ce qu un token ?', mock_response=mock)}")

normal  → Un token est un morceau de mot.
429     → ⏳ Le service est surchargé, réessayez dans quelques instants.
timeout → ⌛ Le modèle met trop de temps à répondre, réessayez.


In [18]:
# Une chaîne vide est "falsy" en Python. Dans certaines versions de LiteLLM, le mock est alors
# IGNORÉ et une vraie requête part vers le fournisseur (facturée si une clé est présente !).
# Démonstration sans risque : on pointe vers un port fermé, avec une fausse clé.
try:
    r = completion(
        model="openai/gpt-4o-mini",
        messages=[{"role": "user", "content": "Qu'est-ce qu'une hallucination ?"}],
        mock_response="",                      # ← on veut simuler une réponse vide…
        api_base="http://127.0.0.1:9/v1",      # port fermé : rien ne peut répondre
        api_key="fausse-cle",
        timeout=2,
        max_retries=0,
    )
    print("✅ Mock respecté : contenu =", repr(r.choices[0].message.content))
except Exception as e:
    print("⚠️ Mock IGNORÉ : LiteLLM a tenté un vrai appel réseau →", type(e).__name__)

# Leçon : un test "mocké" doit AUSSI couper le réseau (on le fera avec pytest-socket au Lab 1).

⚠️ Mock IGNORÉ : LiteLLM a tenté un vrai appel réseau → Timeout


## 2. Observer : tracer les appels avec Langfuse

Crée un compte sur Langfuse Cloud EU (https://cloud.langfuse.com), ou utilise ton instance. Crée ensuite un projet, puis des clés dans Settings → API Keys.
Crée un fichier .env à la racine du repo :

LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_BASE_URL=https://cloud.langfuse.com

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv

# find_dotenv(usecwd=True) remonte les dossiers depuis le notebook jusqu'à trouver le .env
chemin_env = find_dotenv(usecwd=True)
assert chemin_env, "Fichier .env introuvable : créez-le à la racine du repo"
load_dotenv(chemin_env, override=True)

manquantes = [v for v in ("LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL") if not os.getenv(v)]
assert not manquantes, f"Variables manquantes dans .env : {manquantes}"

from langfuse import get_client  # importé APRÈS le chargement du .env

langfuse = get_client()  # client unique, configuré par les variables d'environnement
assert langfuse.auth_check(), "Clés refusées : vérifiez la paire pk/sk et LANGFUSE_BASE_URL"

print("✓ Connecté à Langfuse :", os.environ["LANGFUSE_BASE_URL"])
print("  Clé publique :", os.environ["LANGFUSE_PUBLIC_KEY"][:10] + "…")  # jamais la clé secrète !

✓ Connecté à Langfuse : https://cloud.langfuse.com
  Clé publique : pk-lf-2ea2…


In [ ]:
# Vocabulaire Langfuse :
#   trace       = une requête de bout en bout (ici : une question posée à l'assistant)
#   observation = une étape dans la trace : "span" (étape quelconque) ou "generation" (appel à un LLM)

PRIX = {"input": 0.15e-6, "output": 0.60e-6}  # $ par token, tarif public de gpt-4o-mini (indicatif)


def appeler_llm(question: str, **options) -> str:
    """Appel LLM tracé comme une "generation" : entrée, sortie, tokens, coût, erreurs."""
    messages = [{"role": "user", "content": question}]
    with langfuse.start_as_current_observation(
        as_type="generation", name="appel-llm", model="gpt-4o-mini", input=messages,
    ) as generation:
        try:
            r = completion(model="openai/gpt-4o-mini", messages=messages, **options)
        except Exception as e:
            generation.update(level="ERROR", status_message=f"{type(e).__name__}: {e}")
            raise  # on trace l'erreur, puis on la laisse remonter : c'est à l'appelant de décider
        texte = r.choices[0].message.content
        generation.update(
            output=texte,
            usage_details={"input": r.usage.prompt_tokens, "output": r.usage.completion_tokens},
            cost_details={"input": r.usage.prompt_tokens * PRIX["input"],
                          "output": r.usage.completion_tokens * PRIX["output"]},
        )
        return texte


question = "Qu'est-ce qu'un LLM ?"
with langfuse.start_as_current_observation(as_type="span", name="ma-premiere-trace",
                                           input={"question": question}) as racine:
    texte = appeler_llm(question, mock_response="Un LLM est un modèle de langage entraîné sur "
                                                "d'immenses volumes de texte pour prédire la suite.")
    racine.update(output=texte)
    trace_id = racine.trace_id

# Le SDK envoie les traces en arrière-plan, par lots. flush() force l'envoi immédiat :
# INDISPENSABLE dans un script court, sinon les dernières traces sont perdues à la sortie du programme.
langfuse.flush()

print("Réponse  :", texte)
print("trace_id :", trace_id, "→ Langfuse > Tracing > « ma-premiere-trace »")

Réponse  : Un LLM est un modèle de langage entraîné sur d'immenses volumes de texte pour prédire la suite.
trace_id : ed4250f31764ef4c532f852b8330b7c1 → Langfuse > Tracing > « ma-seconde-trace »


In [ ]:
# 🧪 À vous :
#  1. Dans Langfuse, ouvrez la trace. Retrouvez l'arbre span → generation, la latence, les tokens et le coût.
#  2. Posez une question 10 fois plus longue et relancez. Le coût affiché change-t-il ?
#     Pourquoi ne faut-il pas faire confiance au coût calculé avec un mock ?

# On voit ma-premiere-trace (span) qui contient appel-llm (generation), avec les messages en entrée, le texte en sortie, les tokens, le coût et la durée de chaque étape.
# Le coût ne change pas. Avec mock_response, les tokens sont fixes et ne dépendent pas du vrai prompt (cf. le 🧪 de la cellule 3). Le dashboard a l'air juste, alors qu'il affiche des chiffres inventés. Pour mesurer des coûts réalistes, il faut un mock qui compte les tokens, ou un vrai modèle.

In [ ]:
import uuid

import openai
from langfuse import observe, propagate_attributes

VERSION_APP = "v1"

REPONSES = {
    "Qu'est-ce qu'un token ?": "Un token est un morceau de texte, environ 4 caractères en français.",
    "Qu'est-ce que le RAG ?": "Le RAG consiste à fournir des documents au LLM pour qu'il réponde à partir d'eux.",
    "Qu'est-ce qu'une hallucination ?": "Une hallucination est une réponse fausse formulée avec assurance.",
}
PANNES = {
    "429": litellm.RateLimitError(message="Quota dépassé (simulé)", llm_provider="openai", model="gpt-4o-mini"),
    "timeout": litellm.Timeout(message="Pas de réponse (simulé)", model="gpt-4o-mini", llm_provider="openai"),
}


@observe(name="assistant-ia")  # crée automatiquement la racine de la trace (entrée = arguments, sortie = retour)
def assistant_ia(question: str, session_id: str, panne: str | None = None) -> str:
    with propagate_attributes(          # attributs recopiés sur toutes les observations de la trace
        session_id=session_id,          # regroupe les échanges d'une même conversation
        user_id="etudiant-01",          # en vrai : un identifiant pseudonymisé (RGPD), jamais un e-mail
        tags=["tp-llmops", "notebook-1"],
        version=VERSION_APP,            # pour comparer avant/après un changement
    ):
        simulation = PANNES[panne] if panne else REPONSES.get(question, "Je ne sais pas.")
        try:
            reponse, succes = appeler_llm(question, mock_response=simulation), 1
        except litellm.RateLimitError:
            reponse, succes = "⏳ Le service est surchargé, réessayez dans quelques instants.", 0
        except litellm.Timeout:
            reponse, succes = "⌛ Le modèle met trop de temps à répondre, réessayez.", 0
        except openai.APIError:
            reponse, succes = "❌ Le service est momentanément indisponible.", 0
        langfuse.score_current_trace(name="succes", value=succes, data_type="BOOLEAN")
        return reponse


SESSION = f"conversation-{uuid.uuid4().hex[:8]}"
conversation = [
    ("Qu'est-ce qu'un token ?", None),
    ("Qu'est-ce que le RAG ?", "429"),        # le fournisseur sature…
    ("Qu'est-ce que le RAG ?", None),         # …l'utilisateur réessaie
    ("Qu'est-ce qu'une hallucination ?", "timeout"),
]
for question, panne in conversation:
    print(f"👤 {question}\n🤖 {assistant_ia(question, session_id=SESSION, panne=panne)}\n")

langfuse.flush()
print("Session à ouvrir dans Langfuse > Sessions :", SESSION)

👤 Qu'est-ce qu'un token ?
🤖 Un token est un morceau de texte, environ 4 caractères en français.

👤 Qu'est-ce que le RAG ?
🤖 ⏳ Le service est surchargé, réessayez dans quelques instants.

👤 Qu'est-ce que le RAG ?
🤖 Le RAG consiste à fournir des documents au LLM pour qu'il réponde à partir d'eux.

👤 Qu'est-ce qu'une hallucination ?
🤖 ⌛ Le modèle met trop de temps à répondre, réessayez.

Session à ouvrir dans Langfuse > Sessions : conversation-82604b56
